# Adding FEniCSx-Based Models to PDEForge

This notebook walks through the process of adding a FEniCSx-based PDE model to PDEForge. We use the classic **flow around a cylinder** problem as our example.

## Why FEniCSx?

PDEForge's core models use **spectral methods** (FFT-based), which are excellent for:
- Periodic boundary conditions
- Simple geometries (rectangles, boxes)
- High accuracy with relatively few grid points

However, spectral methods struggle with:
- Complex geometries (cylinders, airfoils, etc.)
- Non-periodic boundary conditions
- Problems with solid boundaries

**FEniCSx** (the modern rewrite of FEniCS) provides:
- Flexible mesh generation via Gmsh
- Support for any boundary condition
- High-level variational form language (UFL)
- Automatic code generation for efficiency

## Prerequisites

To run FEniCSx models, you need:

```bash
conda install -c conda-forge fenics-dolfinx mpich petsc4py gmsh python-gmsh
```

In [ ]:
# Check if FEniCSx is available
try:
    import dolfinx
    print(f"FEniCSx version: {dolfinx.__version__}")
    HAS_FENICSX = True
except ImportError:
    print("FEniCSx not installed. Install with:")
    print("  conda install -c conda-forge fenics-dolfinx mpich petsc4py gmsh python-gmsh")
    HAS_FENICSX = False

## Part 1: Understanding the Problem

### Flow Around a Cylinder

This is a classic benchmark problem in fluid dynamics. A viscous fluid flows past a circular cylinder, creating interesting flow patterns.

![Flow around cylinder](https://upload.wikimedia.org/wikipedia/commons/thumb/d/d4/Karman_vortex_street.gif/330px-Karman_vortex_street.gif)

**Physics:** The governing equations depend on the Reynolds number:

$$Re = \frac{\rho U D}{\mu}$$

where $U$ is inlet velocity, $D$ is cylinder diameter, $\mu$ is viscosity, $\rho$ is density.

- **$Re < 1$**: Stokes flow (creeping flow, symmetric)
- **$1 < Re < 40$**: Steady laminar flow (asymmetric wake)
- **$40 < Re < 200$**: Periodic vortex shedding (von Kármán street)
- **$Re > 200$**: Turbulent wake

For this example, we'll implement **steady Navier-Stokes** for moderate $Re$.

### Equations

**Steady Navier-Stokes:**

$$\rho (\mathbf{u} \cdot \nabla) \mathbf{u} - \mu \nabla^2 \mathbf{u} + \nabla p = 0$$
$$\nabla \cdot \mathbf{u} = 0$$

**Boundary Conditions:**
- Inlet: Parabolic velocity profile
- Outlet: Zero-stress (do-nothing BC)
- Walls: No-slip ($\mathbf{u} = 0$)
- Cylinder: No-slip ($\mathbf{u} = 0$)

### Operator Learning Task

$$\text{inlet velocity scale} \rightarrow (u, v, p)$$

We vary the inlet velocity magnitude and learn to predict the resulting flow field.

## Part 2: Mesh Generation with Gmsh

FEniCSx uses Gmsh for mesh generation. Let's create a mesh for our cylinder flow domain.

In [ ]:
if HAS_FENICSX:
    import numpy as np
    import matplotlib.pyplot as plt
    from mpi4py import MPI
    import gmsh
    from dolfinx.io import gmshio
    from dolfinx import plot
    
    # Geometry parameters
    L = 2.2      # Channel length
    H = 0.41     # Channel height
    cx, cy = 0.2, 0.2  # Cylinder center
    r = 0.05     # Cylinder radius
    resolution = 0.02  # Mesh size
    
    print(f"Domain: [{0}, {L}] x [{0}, {H}]")
    print(f"Cylinder: center=({cx}, {cy}), radius={r}")

In [ ]:
if HAS_FENICSX:
    # Use PDEForge's mesh generation utility
    import sys
    sys.path.insert(0, '..')
    
    from pdeforge.solvers.fenics_utils import create_rectangle_with_hole
    
    mesh = create_rectangle_with_hole(
        L=L, H=H,
        cx=cx, cy=cy, r=r,
        resolution=resolution,
        comm=MPI.COMM_WORLD,
    )
    
    print(f"Mesh created with {mesh.topology.index_map(2).size_local} cells")

In [ ]:
if HAS_FENICSX:
    # Visualize the mesh
    try:
        import pyvista
        pyvista.set_jupyter_backend('static')
        
        from dolfinx.plot import vtk_mesh
        
        topology, cell_types, geometry = vtk_mesh(mesh, mesh.topology.dim)
        grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)
        
        plotter = pyvista.Plotter()
        plotter.add_mesh(grid, show_edges=True, color='white')
        plotter.view_xy()
        plotter.show()
    except ImportError:
        print("PyVista not available for mesh visualization")
        print("Install with: pip install pyvista")

## Part 3: Solving the Navier-Stokes Equations

Now let's implement the FEM solver step by step.

In [ ]:
if HAS_FENICSX:
    from dolfinx import fem
    from dolfinx.fem.petsc import LinearProblem, NonlinearProblem
    from dolfinx.nls.petsc import NewtonSolver
    import ufl
    from ufl import inner, grad, div, dx
    import basix
    from petsc4py import PETSc
    
    # Physical parameters
    mu = 0.001   # Dynamic viscosity
    rho = 1.0    # Density
    U_mean = 0.3 # Mean inlet velocity
    
    # Reynolds number
    Re = rho * U_mean * (2*r) / mu
    print(f"Reynolds number: {Re:.1f}")

In [ ]:
if HAS_FENICSX:
    # Create function spaces (Taylor-Hood: P2-P1)
    P2 = basix.ufl.element("Lagrange", mesh.basix_cell(), 2, shape=(2,))
    P1 = basix.ufl.element("Lagrange", mesh.basix_cell(), 1)
    TH = basix.ufl.mixed_element([P2, P1])
    
    W = fem.functionspace(mesh, TH)
    V, _ = W.sub(0).collapse()  # Velocity space
    
    print(f"Velocity DOFs: {V.dofmap.index_map.size_local * 2}")
    print(f"Total DOFs: {W.dofmap.index_map.size_local}")

In [ ]:
if HAS_FENICSX:
    # Define boundary conditions
    facet_tags = mesh.facet_tags
    
    # Inlet: parabolic profile u(y) = 4 * U_max * y * (H-y) / H^2
    U_max = 1.5 * U_mean
    
    def inlet_velocity(x):
        values = np.zeros((2, x.shape[1]))
        values[0] = 4 * U_max * x[1] * (H - x[1]) / (H ** 2)
        return values
    
    def no_slip(x):
        return np.zeros((2, x.shape[1]))
    
    # Create BC functions
    inlet_func = fem.Function(V)
    inlet_func.interpolate(inlet_velocity)
    
    noslip_func = fem.Function(V)
    noslip_func.interpolate(no_slip)
    
    # Locate boundary DOFs
    fdim = mesh.topology.dim - 1
    
    inlet_dofs = fem.locate_dofs_topological((W.sub(0), V), fdim, facet_tags.find(1))
    wall_dofs = fem.locate_dofs_topological((W.sub(0), V), fdim, facet_tags.find(3))
    cylinder_dofs = fem.locate_dofs_topological((W.sub(0), V), fdim, facet_tags.find(4))
    
    bcs = [
        fem.dirichletbc(inlet_func, inlet_dofs, W.sub(0)),
        fem.dirichletbc(noslip_func, wall_dofs, W.sub(0)),
        fem.dirichletbc(noslip_func, cylinder_dofs, W.sub(0)),
    ]
    
    print(f"Boundary conditions set: inlet, walls, cylinder")

In [ ]:
if HAS_FENICSX:
    # Define variational problem
    w = fem.Function(W)
    (u, p) = ufl.split(w)
    (v, q) = ufl.TestFunctions(W)
    
    # Navier-Stokes weak form
    F = (
        rho * inner(grad(u) * u, v) * dx  # Convection
        + mu * inner(grad(u), grad(v)) * dx  # Diffusion
        - p * div(v) * dx  # Pressure gradient
        - q * div(u) * dx  # Incompressibility
    )
    
    print("Variational form defined")

In [ ]:
if HAS_FENICSX:
    # Solve with Newton's method
    problem = NonlinearProblem(F, w, bcs=bcs)
    solver = NewtonSolver(MPI.COMM_WORLD, problem)
    solver.convergence_criterion = "incremental"
    solver.rtol = 1e-6
    solver.max_it = 50
    
    print("Solving...")
    n_iters, converged = solver.solve(w)
    print(f"Newton solver: {n_iters} iterations, converged={converged}")

In [ ]:
if HAS_FENICSX:
    # Extract solution
    u_sol = w.sub(0).collapse()
    p_sol = w.sub(1).collapse()
    
    # Compute velocity magnitude
    V_scalar = fem.functionspace(mesh, ("Lagrange", 1))
    u_magnitude = fem.Function(V_scalar)
    
    u_vals = u_sol.x.array.reshape(-1, 2)
    u_mag_vals = np.sqrt(u_vals[:, 0]**2 + u_vals[:, 1]**2)
    
    print(f"Max velocity magnitude: {u_mag_vals.max():.4f}")
    print(f"Pressure range: [{p_sol.x.array.min():.4f}, {p_sol.x.array.max():.4f}]")

## Part 4: Interpolating to Regular Grid

For machine learning, we need the solution on a **regular grid**, not the unstructured FEM mesh. PDEForge's `FEniCSModel` base class handles this.

In [ ]:
if HAS_FENICSX:
    from dolfinx import geometry
    
    # Create regular grid
    nx, ny = 128, 64
    x = np.linspace(0, L, nx)
    y = np.linspace(0, H, ny)
    X, Y = np.meshgrid(x, y, indexing='ij')
    points = np.column_stack([X.ravel(), Y.ravel()])
    
    print(f"Regular grid: {nx} x {ny} = {nx*ny} points")

In [ ]:
if HAS_FENICSX:
    # Find cells containing each grid point
    bb_tree = geometry.bb_tree(mesh, mesh.topology.dim)
    cell_candidates = geometry.compute_collisions_points(bb_tree, points)
    cell_collisions = geometry.compute_colliding_cells(mesh, cell_candidates, points)
    
    # Evaluate solution at grid points
    u_grid = np.zeros((len(points), 2))
    p_grid = np.zeros(len(points))
    mask = np.zeros(len(points), dtype=bool)
    
    valid_points = []
    valid_cells = []
    valid_indices = []
    
    for i, point in enumerate(points):
        cells = cell_collisions.links(i)
        if len(cells) > 0:
            valid_points.append(point)
            valid_cells.append(cells[0])
            valid_indices.append(i)
            mask[i] = True
    
    print(f"Valid grid points: {len(valid_indices)} / {len(points)}")
    print(f"Points inside cylinder: {len(points) - len(valid_indices)}")

In [ ]:
if HAS_FENICSX and len(valid_points) > 0:
    valid_points = np.array(valid_points)
    points_3d = np.column_stack([valid_points, np.zeros(len(valid_points))])
    
    # Evaluate velocity and pressure
    u_vals = u_sol.eval(points_3d, valid_cells)
    p_vals = p_sol.eval(points_3d, valid_cells)
    
    for idx, (u_val, p_val) in zip(valid_indices, zip(u_vals, p_vals)):
        u_grid[idx] = u_val
        p_grid[idx] = p_val
    
    # Reshape to grid
    u_grid = u_grid.reshape(nx, ny, 2)
    p_grid = p_grid.reshape(nx, ny)
    mask = mask.reshape(nx, ny)
    
    print(f"Solution interpolated to {nx}x{ny} grid")

In [ ]:
if HAS_FENICSX:
    # Visualize the solution
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    
    # Velocity magnitude
    u_mag = np.sqrt(u_grid[:,:,0]**2 + u_grid[:,:,1]**2)
    u_mag_masked = np.ma.masked_where(~mask, u_mag)
    
    im0 = axes[0,0].pcolormesh(x, y, u_mag_masked.T, cmap='jet', shading='auto')
    axes[0,0].set_title('Velocity Magnitude |u|')
    axes[0,0].set_aspect('equal')
    plt.colorbar(im0, ax=axes[0,0])
    
    # Add cylinder
    circle = plt.Circle((cx, cy), r, color='white', ec='black')
    axes[0,0].add_patch(circle)
    
    # u component
    u_masked = np.ma.masked_where(~mask, u_grid[:,:,0])
    im1 = axes[0,1].pcolormesh(x, y, u_masked.T, cmap='RdBu_r', shading='auto')
    axes[0,1].set_title('u (x-velocity)')
    axes[0,1].set_aspect('equal')
    plt.colorbar(im1, ax=axes[0,1])
    circle = plt.Circle((cx, cy), r, color='gray', ec='black')
    axes[0,1].add_patch(circle)
    
    # v component
    v_masked = np.ma.masked_where(~mask, u_grid[:,:,1])
    im2 = axes[1,0].pcolormesh(x, y, v_masked.T, cmap='RdBu_r', shading='auto')
    axes[1,0].set_title('v (y-velocity)')
    axes[1,0].set_aspect('equal')
    plt.colorbar(im2, ax=axes[1,0])
    circle = plt.Circle((cx, cy), r, color='gray', ec='black')
    axes[1,0].add_patch(circle)
    
    # Pressure
    p_masked = np.ma.masked_where(~mask, p_grid)
    im3 = axes[1,1].pcolormesh(x, y, p_masked.T, cmap='viridis', shading='auto')
    axes[1,1].set_title('Pressure p')
    axes[1,1].set_aspect('equal')
    plt.colorbar(im3, ax=axes[1,1])
    circle = plt.Circle((cx, cy), r, color='white', ec='black')
    axes[1,1].add_patch(circle)
    
    plt.tight_layout()
    plt.show()

## Part 5: Registering the Model with PDEForge

Now let's see how this is integrated into PDEForge's framework.

In [ ]:
import sys
sys.path.insert(0, '..')

from pdeforge import list_models

print("Available models:")
for model in list_models():
    print(f"  - {model}")

In [ ]:
if HAS_FENICSX and 'cylinder_flow_2d' in list_models():
    from pdeforge import generate_dataset
    
    # Generate a small dataset using the unified API
    dataset = generate_dataset(
        model="cylinder_flow_2d",
        n_samples=3,  # Small for demo
        resolution={"x": 64, "y": 32},
        params={
            "viscosity": 0.001,
            "inlet_velocity": 0.3,
            "mesh_resolution": 0.03,  # Coarser for speed
        },
        seed=42,
    )
    
    print(dataset)

## Part 6: The FEniCSModel Base Class

PDEForge provides `FEniCSModel`, which extends `PDEModel` for FEM-based solvers. Here's the key structure:

```python
from pdeforge.core.fenics_base import FEniCSModel
from pdeforge.core.registry import register_model

@register_model("my_fem_model")
class MyFEMModel(FEniCSModel):
    
    NDIM = 2
    DEFAULT_PARAMS = {...}
    INPUT_NAMES = ["input_field"]
    OUTPUT_NAMES = ["u", "v", "p"]
    
    def create_mesh(self):
        """Create or load the computational mesh."""
        # Use Gmsh or load external mesh
        return mesh
    
    def create_function_spaces(self):
        """Define FEM function spaces."""
        self.V = fem.functionspace(self.mesh, ...)
        self.Q = fem.functionspace(self.mesh, ...)
    
    def solve(self, input_data):
        """Solve the PDE and return solution on regular grid."""
        # 1. Setup BCs based on input_data
        # 2. Solve variational problem
        # 3. Interpolate to regular grid
        solution = self.interpolate_to_grid(fem_solution)
        return solution
    
    def generate_ic(self, generator, generator_params, seed):
        """Generate random inputs."""
        return random_input
```

The base class provides:
- `interpolate_to_grid()`: Convert FEM solution to regular arrays
- `create_mask()`: Identify points inside/outside the domain
- Consistent API with spectral models

## Part 7: Adding Your Own FEniCSx Model

### Step-by-Step Guide

1. **Create a new file** in `pdeforge/models/`, e.g., `my_flow.py`

2. **Import the base class and register decorator:**
```python
from pdeforge.core.fenics_base import FEniCSModel
from pdeforge.core.registry import register_model
```

3. **Define your model class:**
```python
@register_model("my_flow_2d")
class MyFlow2D(FEniCSModel):
    ...
```

4. **Implement required methods:**
   - `create_mesh()`: Generate your geometry
   - `create_function_spaces()`: Define FEM spaces
   - `solve()`: Implement your solver
   - `generate_ic()`: Generate random inputs

5. **Register the model** by importing it in `pdeforge/models/__init__.py`

6. **Test** your model works with the unified API:
```python
dataset = generate_dataset(
    model="my_flow_2d",
    n_samples=10,
    resolution={"x": 64, "y": 64},
)
```

## Summary

In this notebook, we covered:

1. **Why FEniCSx** - For complex geometries and non-periodic BCs
2. **Mesh generation** - Using Gmsh via PDEForge utilities
3. **FEM formulation** - Weak forms, function spaces, BCs
4. **Grid interpolation** - Converting FEM solutions to regular arrays
5. **PDEForge integration** - Using `FEniCSModel` base class
6. **Contributing** - Steps to add your own FEniCSx model

The key insight is that **the user-facing API remains the same**:

```python
# Spectral model
dataset = generate_dataset(model="stokes_2d", ...)

# FEniCSx model - same API!
dataset = generate_dataset(model="cylinder_flow_2d", ...)
```

This consistency is what makes PDEForge powerful for operator learning research.